# DL Model for echocardiogram synthetic data generation

## Encoder-decoder training

Importar las librerías necesarias

In [356]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [357]:
import lpips

In [358]:
import cv2

In [359]:
import os
from pathlib import Path
import shutil

Herramienta para procesar videos en frames

In [360]:
def extract_frames_from_avi(video_path, output_folder):
    """
    Extract all frames from an AVI video file and save them as PNG images.
    
    Args:
        video_path (str): Path to the .avi video file
        output_folder (str): Folder where frames will be saved
        
    Returns:
        int: Number of frames extracted
    """
    # Create output folder if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        raise ValueError(f"Could not open video file: {video_path}")
    
    frame_count = 0
    video_name = Path(video_path).stem  # Get filename without extension
    
    while True:
        ret, frame = cap.read()
        
        if not ret:
            break
        
        # Save the frame as PNG
        frame_filename = os.path.join(output_folder, f"{video_name}_frame_{frame_count:06d}.png")
        cv2.imwrite(frame_filename, frame)
        frame_count += 1
    
    cap.release()
    print(f"Extracted {frame_count} frames from {video_path} to {output_folder}")
    
    return frame_count

In [361]:
def process_avi_folder(input_folder, output_base_folder):
    """
    Process all .avi files in a folder, extracting frames from each video.
    Creates a subfolder for each video's frames.
    
    Args:
        input_folder (str): Folder containing .avi files
        output_base_folder (str): Base folder where subfolders for each video will be created
    
    Returns:
        dict: Dictionary with video names as keys and frame counts as values
    """
    Path(output_base_folder).mkdir(parents=True, exist_ok=True)
    
    # Find all .avi files in the input folder
    avi_files = sorted(Path(input_folder).glob("*.avi"))
    
    if not avi_files:
        print(f"No .avi files found in {input_folder}")
        return {}
    
    results = {}
    
    for video_file in avi_files:
        video_name = video_file.stem
        output_folder = os.path.join(output_base_folder, video_name)
        
        try:
            frame_count = extract_frames_from_avi(str(video_file), output_folder)
            results[video_name] = frame_count
        except Exception as e:
            print(f"Error processing {video_file}: {e}")
            results[video_name] = None
    
    print(f"\nProcessing complete! Summary:")
    for video_name, count in results.items():
        if count is not None:
            print(f"  {video_name}: {count} frames")
        else:
            print(f"  {video_name}: Failed")
    
    return results

In [362]:
def consolidate_images_from_subfolders(input_folder, output_folder, move=False):
    """
    Consolidate all images from a folder with subfolders into a single folder.
    
    This function recursively searches through all subfolders and collects all image files
    (jpg, jpeg, png, bmp, tiff, gif) into a single output folder.
    
    Args:
        input_folder (str): Base folder containing subfolders with images
        output_folder (str): Destination folder where all images will be collected
        move (bool): If True, move files (cut). If False, copy files. Default is False.
    
    Returns:
        dict: Summary with 'total_files' count and 'files_by_subfolder' breakdown
    """
    # Define supported image extensions
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif', '.JPG', '.JPEG', '.PNG', '.BMP', '.TIFF', '.GIF'}
    
    # Create output folder if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    input_path = Path(input_folder)
    
    if not input_path.exists():
        raise ValueError(f"Input folder does not exist: {input_folder}")
    
    total_files = 0
    files_by_subfolder = {}
    
    # Recursively search for all image files
    for file_path in input_path.rglob('*'):
        if file_path.is_file() and file_path.suffix in image_extensions:
            # Get the relative subfolder path for tracking
            relative_path = file_path.relative_to(input_path)
            subfolder = str(relative_path.parent)
            
            # Track files by subfolder
            if subfolder not in files_by_subfolder:
                files_by_subfolder[subfolder] = 0
            files_by_subfolder[subfolder] += 1
            
            # Create destination filename (preserve original name)
            output_file = os.path.join(output_folder, file_path.name)
            
            # Handle duplicate filenames by appending underscore + counter
            counter = 1
            base_name = file_path.stem
            extension = file_path.suffix
            while os.path.exists(output_file):
                output_file = os.path.join(output_folder, f"{base_name}_{counter}{extension}")
                counter += 1
            
            # Copy or move the file
            if move:
                shutil.move(str(file_path), output_file)
            else:
                shutil.copy2(str(file_path), output_file)
            
            total_files += 1
    
    # Print summary
    print(f"{'Moved' if move else 'Copied'} {total_files} image files to {output_folder}")
    if files_by_subfolder:
        print("\nFiles by subfolder:")
        for subfolder, count in sorted(files_by_subfolder.items()):
            print(f"  {subfolder}: {count} files")
    
    return {
        'total_files': total_files,
        'files_by_subfolder': files_by_subfolder,
        'move': move
    }

### Clases de redes

In [363]:
class VQGAN(nn.Module):
    def __init__(self, vocab_size=1024, embed_dim=256, beta=0.25, entropy_weight=0.02, temp=1.0):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.beta = beta
        self.entropy_weight = entropy_weight
        self.temp = temp

        # Encoder: Reducción f=16 (256x256 -> 16x16)
        # Usamos la lógica de ConvNeXt simplificada para consistencia
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, 4, stride=4, padding=0),
            nn.ReLU(),
            nn.Conv2d(64, 128, 2, stride=2, padding=0),
            nn.ReLU(),
            nn.Conv2d(128, embed_dim, 1) # Proyección al espacio latente
        )

        self.codebook = nn.Embedding(vocab_size, embed_dim)
        nn.init.uniform_(self.codebook.weight, -1.0, 1.0)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 128, 2, stride=2, padding=0),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=4, padding=0),
            nn.ReLU(),
            nn.Conv2d(64, 1, 3, padding=1),
            nn.Tanh()
        )

    def quantize(self, z):
        # z: [B, C, H, W]
        z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, z.shape[1])

        # Distancia Euclidiana: (a-b)^2 = a^2 + b^2 - 2ab
        d = torch.sum(z_flattened**2, dim=1, keepdim=True) + \
            torch.sum(self.codebook.weight**2, dim=1) - \
            2 * torch.matmul(z_flattened, self.codebook.weight.t())

        indices = torch.argmin(d, dim=1)
        z_q = self.codebook(indices).view(z.shape[0], z.shape[2], z.shape[3], z.shape[1])
        z_q = z_q.permute(0, 3, 1, 2).contiguous()

        codebook_loss = F.mse_loss(z_q, z.detach())
        commitment_loss = F.mse_loss(z, z_q.detach())
        vq_loss = codebook_loss + self.beta * commitment_loss

        # Entropía suave para reducir colapso de código (token collapse)
        soft_assign = F.softmax(-d / self.temp, dim=1)
        avg_probs = soft_assign.mean(dim=0)
        entropy = -torch.sum(avg_probs * torch.log(avg_probs + 1e-10))
        max_entropy = torch.log(torch.tensor(float(self.vocab_size), device=z.device, dtype=avg_probs.dtype))
        entropy_loss = max_entropy - entropy
        perplexity = torch.exp(entropy)

        # Straight-through estimator (STE)
        z_q_ste = z + (z_q - z).detach()
        return z_q_ste, indices, z_q, vq_loss, entropy_loss, perplexity

    def forward(self, x):
        z_e = self.encoder(x)
        z_q_ste, indices, z_q, vq_loss, entropy_loss, perplexity = self.quantize(z_e)
        x_hat = self.decoder(z_q_ste)
        return x_hat, indices, z_e, z_q, vq_loss, entropy_loss, perplexity

class VQGANEncoderBalanced(nn.Module):
    def __init__(self, vocab_size=1024, embed_dim=256, beta=0.25, entropy_weight=0.025, temp=1.0, target_perplexity=192.0, usage_momentum=0.99):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.beta = beta
        self.entropy_weight = entropy_weight
        self.temp = temp
        self.target_perplexity = float(target_perplexity)
        self.usage_momentum = usage_momentum

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, 4, stride=4, padding=0),
            nn.ReLU(),
            nn.Conv2d(64, 128, 2, stride=2, padding=0),
            nn.ReLU(),
            nn.Conv2d(128, embed_dim, 1)
        )

        self.codebook = nn.Embedding(vocab_size, embed_dim)
        nn.init.uniform_(self.codebook.weight, -1.0, 1.0)
        self.register_buffer("usage_ema", torch.full((vocab_size,), 1.0 / vocab_size))

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 128, 2, stride=2, padding=0),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=4, padding=0),
            nn.ReLU(),
            nn.Conv2d(64, 1, 3, padding=1),
            nn.Tanh()
        )

    def set_target_perplexity(self, target_value):
        target_value = float(target_value)
        self.target_perplexity = max(1.0, min(target_value, float(self.vocab_size)))

    def quantize(self, z):
        z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, z.shape[1])
        z_norm = F.normalize(z_flattened, p=2, dim=1)
        codebook_norm = F.normalize(self.codebook.weight, p=2, dim=1)
        d = 2.0 - 2.0 * torch.matmul(z_norm, codebook_norm.t())

        indices = torch.argmin(d, dim=1)
        z_q = self.codebook(indices).view(z.shape[0], z.shape[2], z.shape[3], z.shape[1])
        z_q = z_q.permute(0, 3, 1, 2).contiguous()

        codebook_loss = F.mse_loss(z_q, z.detach())
        commitment_loss = F.mse_loss(z, z_q.detach())
        vq_loss = codebook_loss + self.beta * commitment_loss

        soft_assign = F.softmax(-d / self.temp, dim=1)
        soft_probs = soft_assign.mean(dim=0)
        hard_probs = F.one_hot(indices, num_classes=self.vocab_size).float().mean(dim=0)
        blended_probs = 0.8 * hard_probs + 0.2 * soft_probs

        if self.training:
            self.usage_ema.mul_(self.usage_momentum).add_((1.0 - self.usage_momentum) * hard_probs.detach())

        entropy_batch = -torch.sum(blended_probs * torch.log(blended_probs + 1e-10))
        entropy_hard = -torch.sum(hard_probs * torch.log(hard_probs + 1e-10))
        target_entropy = torch.log(torch.tensor(self.target_perplexity, device=z.device, dtype=blended_probs.dtype))
        entropy_loss = F.relu(target_entropy - entropy_batch)
        perplexity = torch.exp(entropy_hard)

        z_q_ste = z + (z_q - z).detach()
        return z_q_ste, indices, z_q, vq_loss, entropy_loss, perplexity

    def forward(self, x):
        z_e = self.encoder(x)
        z_q_ste, indices, z_q, vq_loss, entropy_loss, perplexity = self.quantize(z_e)
        x_hat = self.decoder(z_q_ste)
        return x_hat, indices, z_e, z_q, vq_loss, entropy_loss, perplexity

In [364]:
# class VQGAN(nn.Module):
#     def __init__(self, vocab_size=1024, embed_dim=256, beta=0.25):
#         super().__init__()
#         self.vocab_size = vocab_size
#         self.embed_dim = embed_dim
#         self.beta = beta # Coeficiente de compromiso

#         # Encoder con Residual Blocks (opcional pero recomendado para eco)
#         self.encoder = nn.Sequential(
#             nn.Conv2d(1, 128, 4, stride=4, padding=0),
#             nn.GroupNorm(32, 128),
#             nn.SiLU(),
#             nn.Conv2d(128, 256, 2, stride=2, padding=0),
#             nn.GroupNorm(32, 256),
#             nn.SiLU(),
#             nn.Conv2d(256, embed_dim, 1)
#         )

#         self.codebook = nn.Embedding(vocab_size, embed_dim)
#         # 1. MEJORA: Inicialización más amplia
#         self.codebook.weight.data.uniform_(-1.0, 1.0)

#         self.decoder = nn.Sequential(
#             nn.ConvTranspose2d(embed_dim, 256, 2, stride=2, padding=0),
#             nn.SiLU(),
#             nn.ConvTranspose2d(256, 128, 4, stride=4, padding=0),
#             nn.SiLU(),
#             nn.Conv2d(128, 1, 3, padding=1),
#             nn.Tanh()
#         )

#     def quantize(self, z):
#         # 2. MEJORA: Normalización L2 (opcional, ayuda mucho a la estabilidad)
#         # z = F.normalize(z, p=2, dim=1) 
        
#         z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, self.embed_dim)
        
#         # Cálculo de distancias
#         d = torch.sum(z_flattened**2, dim=1, keepdim=True) + \
#             torch.sum(self.codebook.weight**2, dim=1) - \
#             2 * torch.matmul(z_flattened, self.codebook.weight.t())
        
#         indices = torch.argmin(d, dim=1)
#         z_q = self.codebook(indices).view(z.shape[0], z.shape[2], z.shape[3], z.shape[1])
#         z_q = z_q.permute(0, 3, 1, 2).contiguous()
        
#         # 3. MEJORA: Pérdida VQ integrada en el forward para limpieza
#         v_loss = F.mse_loss(z_q.detach(), z)
#         c_loss = F.mse_loss(z_q, z.detach())
#         loss = v_loss + self.beta * c_loss
        
#         # STE
#         z_q = z + (z_q - z).detach()
#         return z_q, indices, loss

#     def forward(self, x):
#         z_e = self.encoder(x)
#         z_q, indices, vq_loss = self.quantize(z_e)
#         x_hat = self.decoder(z_q)
#         return x_hat, indices, vq_loss, z_e # Devolvemos vq_loss directamente

In [365]:
class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        def block(in_f, out_f, stride=2):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, 4, stride=stride, padding=1),
                nn.BatchNorm2d(out_f),
                nn.LeakyReLU(0.2, inplace=True)
            )
        self.model = nn.Sequential(
            block(in_channels, 64),
            block(64, 128),
            block(128, 256),
            nn.Conv2d(256, 1, 4, padding=1) # Salida Patch 30x30 aprox
        )

    def forward(self, x):
        return self.model(x)

In [366]:
class VQGANLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # 'vgg' es el estándar porque es muy sensible a texturas médicas
        self.perceptual_loss = lpips.LPIPS(net='vgg').eval() 
        
        # Congelamos la red LPIPS (no queremos entrenarla, solo usarla de juez)
        for param in self.perceptual_loss.parameters():
            param.requires_grad = False

    def forward(self, x, x_hat):
        # x y x_hat deben estar en rango [-1, 1]
        return self.perceptual_loss(x, x_hat).mean()

### Entrenamiento

In [367]:
from tqdm import tqdm

In [368]:
# TRAINING (No-GAN ablation)
USE_DISCRIMINATOR = False
LEARNING_RATE = 0.5e-4
DISC_LR_RATIO = 0.3
GAN_WARMUP_EPOCHS = 12
GAN_RAMP_EPOCHS = 8
GAN_LOSS_WEIGHT = 0.08
PERCEPTUAL_LOSS_WEIGHT = 1.0
TOKEN_ENTROPY_WEIGHT = 0.025
TOKEN_ENTROPY_TEMP = 1.0
TOKEN_TARGET_START = 192
TOKEN_TARGET_END = 420
TOKEN_TARGET_WARMUP_EPOCHS = 4
TOKEN_TARGET_RAMP_EPOCHS = 40
ADAPTIVE_GAN_GAIN = 25.0
ADAPTIVE_GAN_WEIGHT_MAX = 1.5
DISC_UPDATE_INTERVAL = 2
DISC_INPUT_NOISE_STD = 0.03
GRAD_CLIP_NORM = 1.0
device = torch.device("mps" if torch.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

In [369]:
model = VQGANEncoderBalanced(
    entropy_weight=TOKEN_ENTROPY_WEIGHT,
    temp=TOKEN_ENTROPY_TEMP,
    target_perplexity=TOKEN_TARGET_START
).to(device)
discriminator = None
perceptual_loss_fn = lpips.LPIPS(net='vgg').to(device).eval()
for param in perceptual_loss_fn.parameters():
    param.requires_grad = False


Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /opt/miniconda3/envs/medim_torch/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth


In [370]:
opt_ae = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.9))
opt_disc = None

In [371]:
def train_epoch(dataloader, epoch_idx):
    model.train()
    discriminator.train()

    disc_factor = GAN_LOSS_WEIGHT if epoch_idx > GAN_WARMUP_EPOCHS else 0.0

    # Configuramos la barra de progreso
    pbar = tqdm(dataloader, desc=f"Epoch {epoch_idx}")

    for x in pbar:
        x = x.to(device)

        # --- PASO 1: AUTOENCODER (EL CREADOR) ---
        opt_ae.zero_grad()
        x_hat, indices, z_e, z_q, vq_loss, entropy_loss, perplexity = model(x)

        # A. Anatomía y Diccionario
        reconstruction_loss = F.l1_loss(x, x_hat)
        perceptual_loss = perceptual_loss_fn(x, x_hat).mean()

        if disc_factor > 0.0:
            logits_fake = discriminator(x_hat)
            generator_loss = -torch.mean(logits_fake)
        else:
            generator_loss = torch.tensor(0.0, device=device)

        # Pérdida Total (Fase 1)
        total_loss_ae = (
            reconstruction_loss
            + PERCEPTUAL_LOSS_WEIGHT * perceptual_loss
            + vq_loss
            + model.entropy_weight * entropy_loss
            + disc_factor * generator_loss
        )
        total_loss_ae.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        opt_ae.step()

        # --- PASO 2: DISCRIMINADOR (EL JUEZ) ---
        if disc_factor > 0.0:
            opt_disc.zero_grad()

            logits_real = discriminator(x)
            logits_fake_d = discriminator(x_hat.detach()) # Detach para seguridad

            # Hinge loss: más estable que BCE para evitar colapso
            loss_real = torch.mean(F.relu(1.0 - logits_real))
            loss_fake = torch.mean(F.relu(1.0 + logits_fake_d))

            total_loss_disc = 0.5 * (loss_real + loss_fake)
            total_loss_disc.backward()
            torch.nn.utils.clip_grad_norm_(discriminator.parameters(), GRAD_CLIP_NORM)
            opt_disc.step()
        else:
            total_loss_disc = torch.tensor(0.0, device=device)

        # ---------------------------------------
        # Actualización de la barra de progreso
        # ---------------------------------------
        usage_hist = torch.bincount(indices.detach().cpu(), minlength=model.vocab_size)
        active_tokens = int((usage_hist > 0).sum().item())

        pbar.set_postfix({
            "AE_Loss": f"{total_loss_ae.item():.4f}",
            "Disc_Loss": f"{total_loss_disc.item():.4f}",
            "Tokens": active_tokens,
            "Perplexity": f"{perplexity.item():.1f}"
        })

def gan_weight_schedule(epoch_idx):
    if epoch_idx <= GAN_WARMUP_EPOCHS:
        return 0.0
    ramp_progress = min(1.0, (epoch_idx - GAN_WARMUP_EPOCHS) / max(1, GAN_RAMP_EPOCHS))
    return GAN_LOSS_WEIGHT * ramp_progress

def calculate_adaptive_gan_weight(base_loss, generator_loss, last_layer):
    rec_grads = torch.autograd.grad(base_loss, last_layer, retain_graph=True)[0]
    gan_grads = torch.autograd.grad(generator_loss, last_layer, retain_graph=True)[0]
    d_weight = ADAPTIVE_GAN_GAIN * torch.norm(rec_grads) / (torch.norm(gan_grads) + 1e-4)
    return torch.clamp(d_weight, 0.0, ADAPTIVE_GAN_WEIGHT_MAX).detach()

def train_epoch_stabilized(dataloader, epoch_idx):
    model.train()
    discriminator.train()

    disc_factor = gan_weight_schedule(epoch_idx)
    ramp_progress = 0.0 if epoch_idx <= GAN_WARMUP_EPOCHS else min(1.0, (epoch_idx - GAN_WARMUP_EPOCHS) / max(1, GAN_RAMP_EPOCHS))
    noise_std = DISC_INPUT_NOISE_STD * (1.0 - ramp_progress)

    pbar = tqdm(dataloader, desc=f"Epoch {epoch_idx}")

    for step_idx, x in enumerate(pbar, start=1):
        x = x.to(device)

        opt_ae.zero_grad()
        x_hat, indices, z_e, z_q, vq_loss, entropy_loss, perplexity = model(x)

        reconstruction_loss = F.l1_loss(x, x_hat)
        perceptual_loss = perceptual_loss_fn(x, x_hat).mean()
        base_loss = (
            reconstruction_loss
            + PERCEPTUAL_LOSS_WEIGHT * perceptual_loss
            + vq_loss
            + model.entropy_weight * entropy_loss
        )

        if disc_factor > 0.0:
            logits_fake = discriminator(x_hat)
            generator_loss = -torch.mean(logits_fake)
            adaptive_weight = calculate_adaptive_gan_weight(base_loss, generator_loss, model.decoder[-2].weight)
        else:
            generator_loss = torch.tensor(0.0, device=device)
            adaptive_weight = torch.tensor(0.0, device=device)

        total_loss_ae = base_loss + disc_factor * adaptive_weight * generator_loss
        total_loss_ae.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        opt_ae.step()

        if disc_factor > 0.0 and step_idx % DISC_UPDATE_INTERVAL == 0:
            opt_disc.zero_grad()

            if noise_std > 0.0:
                x_real_disc = torch.clamp(x + noise_std * torch.randn_like(x), -1.0, 1.0)
                x_fake_disc = torch.clamp(x_hat.detach() + noise_std * torch.randn_like(x_hat), -1.0, 1.0)
            else:
                x_real_disc = x
                x_fake_disc = x_hat.detach()

            logits_real = discriminator(x_real_disc)
            logits_fake_d = discriminator(x_fake_disc)

            loss_real = torch.mean(F.relu(1.0 - logits_real))
            loss_fake = torch.mean(F.relu(1.0 + logits_fake_d))
            total_loss_disc = 0.5 * (loss_real + loss_fake)
            total_loss_disc.backward()
            torch.nn.utils.clip_grad_norm_(discriminator.parameters(), GRAD_CLIP_NORM)
            opt_disc.step()
        else:
            total_loss_disc = torch.tensor(0.0, device=device)

        usage_hist = torch.bincount(indices.detach().cpu(), minlength=model.vocab_size)
        active_tokens = int((usage_hist > 0).sum().item())

        pbar.set_postfix({
            "AE_Loss": f"{total_loss_ae.item():.4f}",
            "Disc_Loss": f"{total_loss_disc.item():.4f}",
            "Tokens": active_tokens,
            "Perplexity": f"{perplexity.item():.1f}",
            "GAN_w": f"{disc_factor:.3f}",
            "Adv_w": f"{adaptive_weight.item():.3f}"
        })

def train_epoch_translation_only(dataloader, epoch_idx):
    model.train()

    pbar = tqdm(dataloader, desc=f"Epoch {epoch_idx} [NoGAN]")

    for x in pbar:
        x = x.to(device)

        opt_ae.zero_grad()
        x_hat, indices, z_e, z_q, vq_loss, entropy_loss, perplexity = model(x)

        reconstruction_loss = F.l1_loss(x, x_hat)
        perceptual_loss = perceptual_loss_fn(x, x_hat).mean()

        total_loss_ae = (
            reconstruction_loss
            + PERCEPTUAL_LOSS_WEIGHT * perceptual_loss
            + vq_loss
            + model.entropy_weight * entropy_loss
        )

        total_loss_ae.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        opt_ae.step()

        usage_hist = torch.bincount(indices.detach().cpu(), minlength=model.vocab_size)
        active_tokens = int((usage_hist > 0).sum().item())

        pbar.set_postfix({
            "AE_Loss": f"{total_loss_ae.item():.4f}",
            "Disc_Loss": "N/A",
            "Tokens": active_tokens,
            "Perplexity": f"{perplexity.item():.1f}"
        })

def token_target_schedule(epoch_idx):
    if epoch_idx <= TOKEN_TARGET_WARMUP_EPOCHS:
        return float(TOKEN_TARGET_START)
    progress = min(1.0, (epoch_idx - TOKEN_TARGET_WARMUP_EPOCHS) / max(1, TOKEN_TARGET_RAMP_EPOCHS))
    return float(TOKEN_TARGET_START + progress * (TOKEN_TARGET_END - TOKEN_TARGET_START))

def train_epoch_translation_only_encoder_fix(dataloader, epoch_idx):
    model.train()

    target_perplexity = token_target_schedule(epoch_idx)
    model.set_target_perplexity(target_perplexity)

    pbar = tqdm(dataloader, desc=f"Epoch {epoch_idx} [EncFix+NoGAN]")

    for x in pbar:
        x = x.to(device)

        opt_ae.zero_grad()
        x_hat, indices, z_e, z_q, vq_loss, entropy_loss, perplexity = model(x)

        reconstruction_loss = F.l1_loss(x, x_hat)
        perceptual_loss = perceptual_loss_fn(x, x_hat).mean()
        total_loss_ae = (
            reconstruction_loss
            + PERCEPTUAL_LOSS_WEIGHT * perceptual_loss
            + vq_loss
            + model.entropy_weight * entropy_loss
        )

        total_loss_ae.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        opt_ae.step()

        usage_hist = torch.bincount(indices.detach().cpu(), minlength=model.vocab_size)
        active_tokens = int((usage_hist > 0).sum().item())

        pbar.set_postfix({
            "AE_Loss": f"{total_loss_ae.item():.4f}",
            "Disc_Loss": "N/A",
            "Tokens": active_tokens,
            "Perplexity": f"{perplexity.item():.1f}",
            "TargetPx": f"{target_perplexity:.0f}"
        })

Procesar los videos en imágenes

In [372]:
# process_avi_folder("EchoNet-Dynamic/some videos", "EchoNet-Dynamic/Videos")

In [373]:
# consolidate_images_from_subfolders("EchoNet-Dynamic/subframes", "EchoNet-Dynamic/sub all frames", move=True)

In [374]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image

In [375]:
from PIL import Image

In [376]:
# 1. Parámetros de ejecución
NUM_EPOCHS = 100
BATCH_SIZE = 32 # Puedes subirlo a 64 si los frames son de 128x128
NUM_WORKERS = 0
SAVE_EVERY = 5 # Guardar checkpoint cada 5 épocas
CHECKPOINT_DIR = "checkpoints_proid"
SAMPLES_DIR = "samples"

In [377]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)

In [378]:
class EchoDataset(Dataset):
    def __init__(self, folder_path, img_size=256):
        self.folder_path = folder_path
        # Listamos todos los archivos de imagen (ajusta las extensiones si es necesario)
        self.image_files = [f for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
        
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(), # Convierte a [0, 1]
            transforms.Normalize(mean=[0.5], std=[0.5]) # Convierte a [-1, 1]
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.folder_path, self.image_files[idx])
        image = Image.open(img_path).convert('L') # Abrir como escala de grises
        return self.transform(image)

In [379]:
path = "EchoNet-Dynamic/sub all frames" #Cambiar carpeta para hacer el entrenamiento con todas las imagenes

In [380]:
device

device(type='mps')

In [381]:
# 3. EL BLOQUE PROTECTOR (Indispensable en Mac)
if __name__ == '__main__':
    
    # Dataset y DataLoader
    dataset = EchoDataset(folder_path=path, img_size=256)
    train_loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == 'cuda'),
        persistent_workers=NUM_WORKERS > 0
    )

In [ ]:
print(f"Iniciando entrenamiento por {NUM_EPOCHS} épocas...")

for epoch in range(1, NUM_EPOCHS + 1):
    # Ejecutamos la función de época que definimos antes
    train_epoch_translation_only_encoder_fix(train_loader, epoch)
    
    # --- Tareas de mantenimiento al final de cada época ---
    
    # A. Guardar pesos del modelo
    if epoch % SAVE_EVERY == 0:
        checkpoint_path = os.path.join(CHECKPOINT_DIR, f"vqgan_heart_ep{epoch}.pth")
        checkpoint_payload = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'disc_state_dict': None,
            'opt_ae_state_dict': opt_ae.state_dict(),
            'opt_disc_state_dict': None,
        }
        if discriminator is not None:
            checkpoint_payload['disc_state_dict'] = discriminator.state_dict()
        if opt_disc is not None:
            checkpoint_payload['opt_disc_state_dict'] = opt_disc.state_dict()
        torch.save(checkpoint_payload, checkpoint_path)
        print(f"Checkpoint guardado en: {checkpoint_path}")

    # B. Generar una muestra visual para control de calidad
    # Esto te permite ver si el modelo está aprendiendo los bordes correctamente
    with torch.no_grad():
        model.eval()
        # Tomamos un batch de ejemplo
        sample_x = next(iter(train_loader)).to(device)
        x_hat, _, _, _, _, _, _ = model(sample_x)
        
        # Guardamos la comparación (Opcional: puedes usar torchvision.utils.save_image)
        # transform.Normalize(mean=[0.5], std=[0.5]) invierte [-1, 1] a [0, 1]
        sample_img = (x_hat[0] + 1) / 2 
        save_image(sample_img, f"{SAMPLES_DIR}/res_ep{epoch}.png")
        model.train()

print("¡Entrenamiento completado!")

Iniciando entrenamiento por 100 épocas...


Epoch 5: 100%|██████████| 132/132 [02:18<00:00,  1.05s/it, AE_Loss=0.8095, Disc_Loss=0.0000, Tokens=104, Perplexity=417.7]


Checkpoint guardado en: checkpoints_proid/vqgan_heart_ep5.pth


Epoch 10: 100%|██████████| 132/132 [02:21<00:00,  1.07s/it, AE_Loss=0.9087, Disc_Loss=0.0000, Tokens=132, Perplexity=413.2]


Checkpoint guardado en: checkpoints_proid/vqgan_heart_ep10.pth


Epoch 15: 100%|██████████| 132/132 [03:06<00:00,  1.42s/it, AE_Loss=1.9912, Disc_Loss=0.2682, Tokens=170, Perplexity=329.4]


Checkpoint guardado en: checkpoints_proid/vqgan_heart_ep15.pth


Epoch 20: 100%|██████████| 132/132 [03:11<00:00,  1.45s/it, AE_Loss=5.0770, Disc_Loss=0.1765, Tokens=191, Perplexity=248.2]


Checkpoint guardado en: checkpoints_proid/vqgan_heart_ep20.pth


Epoch 25:  86%|████████▋ | 114/132 [02:52<00:27,  1.55s/it, AE_Loss=11.9325, Disc_Loss=0.0449, Tokens=184, Perplexity=172.2]